# 🧬 BioHub Cell Tracking — Scoring Metric & Data, Fully Explained

The two questions everyone new to this competition asks are:

1. **"What does my score actually mean?"** — the metric mixes an *adjusted edge Jaccard* with a *division Jaccard*, and the node-count adjustment surprises people.
2. **"What does the data even look like?"** — 3D volumes over time, anisotropic voxels, and sparse lineage ground truth.

This notebook answers both, from the ground up, with **runnable** code and plots. Everything here is derived from the public competition metric and the public data — no magic, no secret post-processing. If it helps you, an upvote is appreciated 🙂

**Contents**
- Part 1 — The scoring metric, term by term (with worked examples)
- Part 2 — Loading the data (images + ground-truth lineage graphs)
- Part 3 — Visual EDA across the training videos
- Part 4 — Practical takeaways

In [ ]:
# Setup — the only third-party reader we need is `zarr` (for the .zarr / .geff data).
# numpy / pandas / matplotlib are already present. This install guard is a no-op
# if zarr is already available in the image.
import importlib, subprocess, sys
try:
    importlib.import_module("zarr")
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zarr>=3.0.10,<4"], check=False)
import zarr
print("zarr", zarr.__version__)


## Part 1 — The scoring metric, term by term

The leaderboard score for one video is:

$$\text{score} = \underbrace{J_{edge}\cdot\left(1 - \alpha\cdot r\right)}_{\text{adjusted edge Jaccard}} \;+\; \beta\cdot J_{div}$$

with $\alpha = 0.1$ and $\beta = 0.1$, where

- $J_{edge} = \dfrac{TP_e}{TP_e + FP_e + FN_e}$ — Jaccard over **edges** (parent→child links between consecutive detections),
- $r = \dfrac{N_{pred} - N_{est}}{N_{est}}$ — the **node-count ratio**: how far your number of predicted nodes is from the estimated true count $N_{est}$,
- $J_{div} = \dfrac{TP_d}{TP_d + FP_d + FN_d}$ — Jaccard over **division events**.

Scores are **micro-averaged**: TP/FP/FN are summed across all videos *before* the Jaccard is taken, so bigger videos carry more weight.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ALPHA = 0.1   # node-ratio penalty coefficient
BETA  = 0.1   # weight of the division Jaccard in the final score

def jaccard(tp, fp, fn):
    denom = tp + fp + fn
    return tp / denom if denom > 0 else float("nan")

def adj_edge_jaccard(edge_j, n_pred, n_est):
    """Edge Jaccard adjusted by how far the node count is from the estimate."""
    node_ratio = (n_pred - n_est) / n_est
    return max(0.0, edge_j * (1.0 - ALPHA * node_ratio)), node_ratio

def total_score(edge_tp, edge_fp, edge_fn, n_pred, n_est,
                div_tp=0, div_fp=0, div_fn=0):
    edge_j = jaccard(edge_tp, edge_fp, edge_fn)
    adj, ratio = adj_edge_jaccard(edge_j, n_pred, n_est)
    div_j = jaccard(div_tp, div_fp, div_fn)
    has_div = (div_tp + div_fp + div_fn) > 0
    score = adj + (BETA * div_j if has_div else 0.0)
    return dict(edge_j=edge_j, node_ratio=ratio, adj_edge_j=adj,
                div_j=(div_j if has_div else float("nan")), score=score)

# A worked example: strong edges, a few divisions, node count slightly under estimate
example = total_score(edge_tp=9000, edge_fp=600, edge_fn=900,
                      n_pred=9700, n_est=10000,
                      div_tp=40, div_fp=8, div_fn=25)
for k, v in example.items():
    print(f"{k:12s} = {v:.4f}")


### The node-count adjustment is the part people miss

Because the edge Jaccard is multiplied by $(1 - 0.1\,r)$:

- If you predict **more** nodes than estimated ($r > 0$) your edge score is **shrunk**.
- If you predict **fewer** nodes than estimated ($r < 0$) the factor is **> 1** — a small bonus.

So node-count *calibration* matters, not just raw edge accuracy. The plot below holds the raw edge Jaccard fixed and sweeps the node ratio.

In [ ]:
edge_j_fixed = 0.90
ratios = np.linspace(-0.4, 0.4, 200)
adj = np.clip(edge_j_fixed * (1 - ALPHA * ratios), 0, None)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(ratios * 100, adj, lw=2.5, color="#2a7de1")
ax.axhline(edge_j_fixed, ls="--", color="grey", label=f"raw edge Jaccard = {edge_j_fixed}")
ax.axvline(0, color="black", lw=0.8)
ax.fill_between(ratios * 100, edge_j_fixed, adj, where=(adj > edge_j_fixed),
                color="#8bd", alpha=0.35, label="bonus zone (predicted < estimated)")
ax.set_xlabel("node-count ratio r  (%)  =  (N_pred - N_est) / N_est")
ax.set_ylabel("adjusted edge Jaccard")
ax.set_title("How the node-count adjustment reshapes the edge score")
ax.legend(); ax.grid(alpha=0.25)
plt.show()


### Division events

A **division** in the ground truth has the shape

```
parent → divider → child A → grandchild A
                 → child B → grandchild B
```

A predicted division is credited (**TP**) when the prediction covers the pre-split stage *and* both daughter lineages of a real division, connected in one component. A predicted fork whose matched ground-truth cell does **not** divide is a **FP**; a real division the prediction misses is a **FN**. Divisions are rare, so $J_{div}$ enters the score at weight $0.1$ — a tie-breaker, but a meaningful one near the top of the board.

## Part 2 — Loading the data

Each video ships as:

- an **image** `\<id\>.zarr` — a 4-D array `(T, Z, Y, X)` of `uint16` intensities, and
- a **ground-truth lineage** `\<id\>.geff` — a small graph (Zarr-v3 based) with node properties `t, z, y, x` and directed `edges` (parent → child).

Voxels are **anisotropic**: physical scale ≈ `(z, y, x) = (1.625, 0.40625, 0.40625)` µm, so distances must be scaled. The helpers below read a `.geff` graph straight from its Zarr arrays — no special tracking library required.

In [ ]:
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd
import zarr

COMPETITION = "biohub-cell-tracking-during-development"

def find_comp_dir():
    # env override lets this notebook be developed anywhere; on Kaggle the
    # standard mounts are tried in order.
    env = os.environ.get("BIOHUB_COMP_DIR", "").strip()
    candidates = ([Path(env)] if env else []) + [
        Path(f"/kaggle/input/competitions/{COMPETITION}"),
        Path(f"/kaggle/input/{COMPETITION}"),
    ]
    for c in candidates:
        if c and c.exists():
            return c
    raise FileNotFoundError("Could not locate the competition data directory")

COMP_DIR = find_comp_dir()
print("competition dir:", COMP_DIR)

def list_gt_graphs(comp_dir):
    """Return {dataset_id: geff_path} for every ground-truth graph we can find."""
    roots = [comp_dir / "train", comp_dir]
    found = {}
    for root in roots:
        if not root.exists():
            continue
        for p in sorted(glob.glob(str(root / "*.geff"))):
            found.setdefault(Path(p).stem, Path(p))
    return found

def read_geff(geff_path):
    """Read a .geff ground-truth graph directly from its zarr arrays."""
    g = zarr.open(str(geff_path), mode="r")
    nodes = pd.DataFrame({
        "node_id": np.asarray(g["nodes/ids"][:]).reshape(-1),
        "t": np.asarray(g["nodes/props/t/values"][:]).reshape(-1),
        "z": np.asarray(g["nodes/props/z/values"][:]).reshape(-1),
        "y": np.asarray(g["nodes/props/y/values"][:]).reshape(-1),
        "x": np.asarray(g["nodes/props/x/values"][:]).reshape(-1),
    })
    e = np.asarray(g["edges/ids"][:])
    edges = pd.DataFrame(e.reshape(-1, 2), columns=["source_id", "target_id"]) if e.size else \
            pd.DataFrame(columns=["source_id", "target_id"])
    return nodes, edges

gt_graphs = list_gt_graphs(COMP_DIR)
print(f"found {len(gt_graphs)} ground-truth graphs:", list(gt_graphs)[:6], "...")


In [ ]:
# Build a per-video summary table straight from the ground-truth graphs.
rows = []
for name, path in gt_graphs.items():
    nodes, edges = read_geff(path)
    out_deg = edges.groupby("source_id").size() if len(edges) else pd.Series(dtype=int)
    n_div = int((out_deg >= 2).sum())
    rows.append({
        "dataset": name,
        "gt_nodes": len(nodes),
        "gt_edges": len(edges),
        "t_min": int(nodes["t"].min()) if len(nodes) else -1,
        "t_max": int(nodes["t"].max()) if len(nodes) else -1,
        "divisions": n_div,
    })
meta = pd.DataFrame(rows).sort_values("gt_nodes", ascending=False).reset_index(drop=True)
print("Ground-truth lineage annotations are SPARSE — a few tracked lineages per video,")
print("not every cell. The metric matches predictions against these annotated tracks.\n")
meta


## Part 3 — Visual EDA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)

# (a) annotated nodes per video
axes[0].bar(range(len(meta)), meta["gt_nodes"], color="#2a7de1")
axes[0].set_title("Annotated GT nodes per video")
axes[0].set_xlabel("video (sorted)"); axes[0].set_ylabel("# nodes")

# (b) divisions per video
axes[1].bar(range(len(meta)), meta["divisions"], color="#e07a2a")
axes[1].set_title("Annotated divisions per video")
axes[1].set_xlabel("video (sorted)"); axes[1].set_ylabel("# divisions (out-degree >= 2)")

# (c) temporal span
axes[2].bar(range(len(meta)), meta["t_max"] - meta["t_min"], color="#3aa76d")
axes[2].set_title("Annotated time span per video")
axes[2].set_xlabel("video (sorted)"); axes[2].set_ylabel("frames spanned")
plt.show()

print(f"totals — videos: {len(meta)} | GT nodes: {meta.gt_nodes.sum()} | "
      f"GT edges: {meta.gt_edges.sum()} | divisions: {meta.divisions.sum()}")


In [ ]:
# Track-length distribution: walk each lineage forward from its roots.
from collections import defaultdict, deque

def track_lengths(nodes, edges):
    if not len(edges):
        return []
    children = defaultdict(list)
    indeg = defaultdict(int)
    for s, t in edges[["source_id", "target_id"]].itertuples(index=False):
        children[s].append(t); indeg[t] += 1
    roots = [n for n in nodes["node_id"] if indeg[n] == 0]
    lengths = []
    for r in roots:
        # longest path length from this root (in nodes)
        seen, q, best = set(), deque([(r, 1)]), 1
        while q:
            u, d = q.popleft(); best = max(best, d)
            for v in children[u]:
                if v not in seen:
                    seen.add(v); q.append((v, d + 1))
        lengths.append(best)
    return lengths

all_lengths = []
for name, path in gt_graphs.items():
    nodes, edges = read_geff(path)
    all_lengths += track_lengths(nodes, edges)

fig, ax = plt.subplots(figsize=(8, 4.2))
if all_lengths:
    ax.hist(all_lengths, bins=min(30, max(5, len(set(all_lengths)))),
            color="#7a4fd1", edgecolor="white")
    ax.set_title(f"Ground-truth lineage lengths  (n={len(all_lengths)} lineages)")
    ax.set_xlabel("lineage length (frames)"); ax.set_ylabel("count")
    ax.grid(alpha=0.25)
plt.show()


In [ ]:
# One image frame: a max-intensity projection over Z, with GT points overlaid.
def open_image(dataset_id):
    for root in [COMP_DIR / "train", COMP_DIR]:
        zp = root / f"{dataset_id}.zarr"
        if zp.exists():
            arr = zarr.open(str(zp), mode="r")
            arr = arr["0"] if hasattr(arr, "keys") and "0" in list(arr.keys()) else arr
            return arr
    return None

demo = meta.iloc[0]["dataset"]
img = open_image(demo)
nodes, edges = read_geff(gt_graphs[demo])

if img is not None:
    T = img.shape[0]
    t0 = int(nodes["t"].median()) if len(nodes) else 0
    t0 = min(max(t0, 0), T - 1)
    vol = np.asarray(img[t0]).astype(np.float32)     # (Z, Y, X)
    proj = vol.max(axis=0)                            # max over Z -> (Y, X)

    pts = nodes[nodes["t"] == t0]
    fig, ax = plt.subplots(figsize=(7.5, 7.5))
    ax.imshow(proj, cmap="gray",
              vmax=np.percentile(proj, 99.5), vmin=np.percentile(proj, 40))
    if len(pts):
        ax.scatter(pts["x"], pts["y"], s=70, facecolors="none",
                   edgecolors="#ff3b3b", linewidths=1.6, label="GT cell")
        ax.legend(loc="upper right")
    ax.set_title(f"{demo}  —  frame t={t0}  (Z max-projection, {vol.shape[0]} slices)")
    ax.set_xlabel("x"); ax.set_ylabel("y")
    plt.show()
    print(f"volume shape (T,Z,Y,X) = {img.shape}, dtype = {img.dtype}")


## Part 4 — Practical takeaways

- **The score is edge-dominated.** The adjusted edge Jaccard is the bulk of it; the division term enters at weight `0.1`.
- **Calibrate your node count.** Because of the $(1 - 0.1\,r)$ factor, wildly over-predicting nodes is quietly penalised, while a node count at or a little below the estimate is not.
- **Respect the voxel anisotropy** (`z` is ~4× coarser than `x,y`) whenever you compute distances for matching or linking.
- **Ground truth is sparse** — the metric scores your predictions against a handful of carefully annotated lineages per video, not every cell.
- **Divisions are rare but decisive at the top** — worth getting right once the edge score plateaus.

That's the whole scoring surface, no secrets. Good luck, and happy tracking! 🧬